# Function Calling and Pydantic Validation

This notebook introduces two mechanisms for enforcing structured output:

1. **Function calling (tool use):** tells the LLM to return a structured object matching a schema you define
2. **Pydantic validation:** applies Python-side type and business rule validation to the returned object

Together they guarantee that every extraction produces a valid, typed record or fails with a handleable error.

---

## 1. Setup

In [11]:
# Load environment variables from the .env file.
# python-dotenv reads the file and makes the key available via os.getenv().

from dotenv import load_dotenv
import os
import anthropic
import json
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal, Optional

load_dotenv()

# Load ANTHROPIC_API_KEY from the project-root .env (one folder up from week1/).
load_dotenv(dotenv_path=os.path.join('..', '.env'))

assert os.environ.get('ANTHROPIC_API_KEY'), (
    'ANTHROPIC_API_KEY not found. Add it to the project-root .env file.'
)

# The client reads ANTHROPIC_API_KEY from the environment automatically.
client = anthropic.Anthropic()

MODEL = 'claude-haiku-4-5'   # supports temperature / top_k / top_p (Opus 4.8 / 4.7 do not)
print(f'Client ready. Using model: {MODEL}')

Client ready. Using model: claude-haiku-4-5


In [12]:
TEST_INPUTS = {
    "T1": (
        "PV team here. We have a serious unexpected SUSAR for study NCT-2244 "
        "and need to report to EMA within 15 days per ICH E2A."
    ),
    "T2": "Hi, we need to file a Type II variation for our cardiovascular product with the EMA.",
    "T3": (
        "CMC Regulatory here. FDA issued a 483 observation on our manufacturing site in Pune. "
        "We have 15 business days to respond."
    ),
    "T4": "Can you check what the weather is like in Mumbai today?",
    "T5": (
        "Labelling team. We need to update the SmPC for our oncology product "
        "following the latest EMA assessment report."
    ),
}

print("Setup complete.")

Setup complete.


## 2. Defining the Tool Schema

The tool schema is a JSON structure that tells the LLM what fields to return and what values are permitted. The API uses this schema to constrain the model's output at generation time — it is not just a soft instruction.

Every enum field lists its allowed values. Required fields are listed in the `required` array.

In [13]:
# Tool schema definition for the five SmartIntake compliance fields.
# This schema is passed directly to the Anthropic API as a tool definition.

INTAKE_TOOL = {
    "name": "extract_intake_fields",
    "description": (
        "Extract the five structured compliance fields from a pharmaceutical regulatory query. "
        "Use null for any field that cannot be determined from the message."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query_type": {
                "type": "string",
                "enum": [
                    "complaint", "submission", "variation", "safety_signal",
                    "label_update", "inspection", "general_enquiry"
                ],
                "description": "The nature of the regulatory interaction."
            },
            "regulation_ref": {
                "type": "string",
                "enum": [
                    "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
                    "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
                ],
                "description": "The primary regulatory framework the query relates to."
            },
            "product_area": {
                "type": "string",
                "enum": [
                    "oncology", "cardiovascular", "infectious_disease",
                    "cmc", "clinical", "labelling", "general"
                ],
                "description": "The therapeutic area or functional domain."
            },
            "urgency": {
                "type": "string",
                "enum": ["routine", "standard", "urgent", "critical"],
                "description": "Deadline-driven priority tier."
            },
            "submitting_team": {
                "type": "string",
                "description": "Name of the submitting team or function. Never an individual name."
            }
        },
        # All five fields are required in the schema, but values can be null
        # when the information is not present in the message.
        # Pydantic will handle the null cases and prompt for missing fields.
        "required": ["query_type", "regulation_ref", "product_area", "urgency", "submitting_team"]
    }
}

print("Tool schema defined.")
print(f"Fields: {list(INTAKE_TOOL['input_schema']['properties'].keys())}")

Tool schema defined.
Fields: ['query_type', 'regulation_ref', 'product_area', 'urgency', 'submitting_team']


## 3. Making a Function Calling API Call

Pass the tool definition to the API. The model returns a `tool_use` content block instead of a text block when it decides to call the tool.

In [14]:
# System prompt for function calling — same constraints as NB-03, no change needed
EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
Use the extract_intake_fields tool to extract the five compliance fields.
Think step by step about the regulatory context before choosing enum values.
NEVER infer urgency from tone — only from explicit deadlines.
NEVER use an individual name as submitting_team.
Use null for fields you cannot determine.
"""


def call_with_tool(user_message: str) -> dict:
    """
    Make a function calling API request and return the extracted tool arguments.
    
    Returns a dict of the five fields, or raises an exception if the model
    did not call the tool.
    """
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        system=EXTRACTION_SYSTEM,
        tools=[INTAKE_TOOL],
        # tool_choice forces the model to use our tool rather than respond in text
        tool_choice={"type": "tool", "name": "extract_intake_fields"},
        messages=[{"role": "user", "content": user_message}]
    )

    # Find the tool_use block in the response content
    for block in response.content:
        if block.type == "tool_use":
            return block.input  # This is already a dict — no JSON parsing needed

    raise ValueError("Model did not return a tool_use block.")


# Test with T3 — all five fields should be present
raw_fields = call_with_tool(TEST_INPUTS["T3"])
print(raw_fields)
print("-----------------------")
print("Extracted fields (T3):")
print(json.dumps(raw_fields, indent=2))

{'query_type': 'inspection', 'regulation_ref': 'FDA_21CFR', 'product_area': 'cmc', 'urgency': 'urgent', 'submitting_team': 'CMC Regulatory'}
-----------------------
Extracted fields (T3):
{
  "query_type": "inspection",
  "regulation_ref": "FDA_21CFR",
  "product_area": "cmc",
  "urgency": "urgent",
  "submitting_team": "CMC Regulatory"
}


## 4. Defining the Pydantic Model

Pydantic validates the extracted dict against the schema and applies custom business rules.

Two custom validators enforce compliance constraints that the tool schema cannot express:
- `submitting_team` must not be empty
- `submitting_team` must not look like an individual's name

In [15]:
# The IntakeRecord Pydantic model.
# This is the canonical schema for the project — copy this to schema.py in your implementation.

class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]
    urgency: Literal["routine", "standard", "urgent", "critical"]
    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v: str) -> str:
        """Reject empty or whitespace-only team names."""
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()  # Normalise capitalisation

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v: str) -> str:
        """
        Heuristic check: reject single capitalised tokens that look like person names.
        
        A real production system would use a NER model here.
        For the capstone, this heuristic catches the most common case.
        
        Example of what this rejects: 'Priya' or 'John'
        Example of what this accepts: 'PV Team' or 'CMC Regulatory'
        """
        tokens = v.strip().split()
        if len(tokens) == 1 and tokens[0][0].isupper() and tokens[0].isalpha():
            raise ValueError(
                "submitting_team appears to be an individual name. "
                "Please provide a team or function name instead."
            )
        return v


print("IntakeRecord model defined.")

IntakeRecord model defined.


## 5. Validating a Successful Extraction

This is a purely local operation. It unpacks the `raw_fields` dict (already returned by Claude) and runs Pydantic's type/validator checks


In [17]:
# Validate the T3 extraction against the Pydantic model.
# T3 provides all five fields, so validation should pass cleanly.

try:
    record = IntakeRecord(**raw_fields)
    print("Validation passed.")
    print(record.model_dump_json(indent=2))
except ValidationError as e:
    print("Validation failed:")
    print(e)

Validation passed.
{
  "query_type": "inspection",
  "regulation_ref": "FDA_21CFR",
  "product_area": "cmc",
  "urgency": "urgent",
  "submitting_team": "Cmc Regulatory"
}


## 6. Handling Validation Errors

T2 provides `query_type`, `regulation_ref`, and `product_area` but omits `urgency` and `submitting_team`. The model will return `null` for those fields. Pydantic will raise a `ValidationError` because `null` is not a valid Literal value.

We catch this error and identify which fields need to be collected.

In [18]:
# T2 is a partial submission — urgency and submitting_team are not in the message.
# Pydantic will raise ValidationError for the missing (null) required fields.

partial_fields = call_with_tool(TEST_INPUTS["T2"])
print("Raw extraction for T2:")
print(json.dumps(partial_fields, indent=2))
print()

Raw extraction for T2:
{
  "query_type": "variation",
  "regulation_ref": "EMA_CTR",
  "product_area": "cardiovascular",
  "urgency": "routine",
  "submitting_team": "UNKNOWN"
}



In [19]:
# Filter out null values before attempting validation
# None values in a Literal field will always fail validation
non_null_fields = {k: v for k, v in partial_fields.items() if v is not None}

print(non_null_fields)
print("-----------------------")


try:
    record = IntakeRecord(**non_null_fields)
    print("All fields present — record is valid.")
except ValidationError as e:
    # Extract the names of the fields that failed validation
    missing_fields = [str(err["loc"][0]) for err in e.errors()]
    print(f"Validation error — missing fields: {missing_fields}")
    print("The session loop should now ask the user for these specific fields.")

{'query_type': 'variation', 'regulation_ref': 'EMA_CTR', 'product_area': 'cardiovascular', 'urgency': 'routine', 'submitting_team': 'UNKNOWN'}
-----------------------
Validation error — missing fields: ['submitting_team']
The session loop should now ask the user for these specific fields.


## 7. The Retry Loop

The retry pattern gives the model two attempts to produce a valid extraction. If both attempts fail, the system falls back to a direct question.

Here we simulate the pattern with a mock user response to show how accumulated fields merge across attempts.

In [ ]:
# Simulate the two-attempt retry pattern.
# In the real session loop this interacts with the user across multiple turns.
# Here we simulate the conversation programmatically.

def collect_missing_fields_interactively(already_collected: dict, missing: list) -> dict:
    """
    Simulate collecting missing fields.
    In the real CLI, this prompts the user with input().
    Here we return hardcoded answers to demonstrate the pattern.
    """
    # Simulated user responses for T2's missing fields
    simulated_answers = {
        "urgency": "standard",
        "submitting_team": "Regulatory Affairs",
    }
    
    result = dict(already_collected)  # Start with what we already have
    for field in missing:
        if field in simulated_answers:
            print(f"[simulated user provides '{field}': {simulated_answers[field]}]")
            result[field] = simulated_answers[field]
    return result


# Begin with the partial T2 extraction
collected = {k: v for k, v in partial_fields.items() if v is not None}
print(f"Initially collected: {list(collected.keys())}")

# Retry loop — up to two attempts before falling back to direct questions
for attempt in range(2):
    try:
        record = IntakeRecord(**collected)
        print(f"\nValidation passed on attempt {attempt + 1}.")
        print(record.model_dump_json(indent=2))
        break
    except ValidationError as e:
        missing = [str(err["loc"][0]) for err in e.errors()]
        print(f"\nAttempt {attempt + 1}: missing fields = {missing}")
        collected = collect_missing_fields_interactively(collected, missing)
else:
    # Both attempts failed — ask directly without the extraction step
    print("\nTwo failed extraction attempts. Switching to direct question mode.")

## 8. Testing the Validator — Individual Names

Verify that the `team_not_a_person` validator fires correctly on a single-token name.

In [ ]:
# Test cases for the submitting_team validator.
# Each case should either pass or raise a specific error.

test_cases = [
    # (submitting_team value, should_pass)
    ("PV Team", True),
    ("CMC Regulatory", True),
    ("Pharmacovigilance", True),   # Single word but not a typical first name — test it
    ("Priya", False),              # Single capitalised alpha token — looks like a name
    ("John", False),               # Same pattern
    ("", False),                   # Empty string
    ("   ", False),                # Whitespace only
]

# We need a complete valid record to test the submitting_team field in isolation.
# The other fields are hardcoded here so we can isolate the team validator.
base_fields = {
    "query_type": "submission",
    "regulation_ref": "FDA_21CFR",
    "product_area": "cmc",
    "urgency": "standard",
}

print(f"{'submitting_team':<25} {'Expected':<10} {'Result':<10}")
print("-" * 50)

for team_value, should_pass in test_cases:
    try:
        IntakeRecord(**base_fields, submitting_team=team_value)
        actual = "PASS"
    except ValidationError:
        actual = "FAIL"
    
    expected = "PASS" if should_pass else "FAIL"
    match = "OK" if actual == expected else "UNEXPECTED"
    display_value = repr(team_value)[:20]
    print(f"{display_value:<25} {expected:<10} {actual:<10} {match}")

---
## Summary

You have:
- Defined the tool schema for the five compliance fields
- Made function calling API requests and parsed the `tool_use` response block
- Defined the `IntakeRecord` Pydantic model with two custom validators
- Handled `ValidationError` and identified missing fields
- Implemented and tested the two-attempt retry loop
- Verified that the `submitting_team` validator fires correctly on individual names

**Next:** Open `NB-05_langchain_chains.ipynb` to wrap this logic in LangChain chains.